In [ ]:
import os
import json
from openai import OpenAI
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    from google.colab import userdata
    # Credentials resolved above
except Exception:
    NVIDIA_API_KEY = os.environ.get('NVIDIA_API_KEY')

# ==============================================================================
# 06_Adversarial_Evaluation_Comparison.ipynb
# Research Framework: Beyond Redaction
# Objective: Exposing Protected Persistent Context to Adversarial Profiler
# Model pin: deepseek-ai/deepseek-r1 (pinned 2026-08-22, replacing deprecated deepseek-v4-pro)
# ==============================================================================

print("[1/2] Loading protected edge context asset from persistent storage...")

protected_payload = None

try:
    # Ensure drive is attached to this notebook session
    if not os.path.exists("/content/drive"):
        pass

    # Explicitly point to the persistent folder used in Notebook 05
    PERSISTENT_CACHE_DIR = os.environ.get("DATA_DIR", "./data")
    TARGET_FILE = os.path.join(PERSISTENT_CACHE_DIR, "protected_generalized_context.txt")

    if os.path.exists(TARGET_FILE):
        with open(TARGET_FILE, 'r') as f:
            protected_payload = f.read()
        print(f"--> Success: Loaded protected context asset ({len(protected_payload.split())} words).")
    else:
        raise FileNotFoundError(f"Target file not found at path: {TARGET_FILE}")

except Exception as e:
    print(f"[CRITICAL ERROR] Protected context vectors missing: {e}")

# Re-initialize the execution client using credentials
print("\nRe-initializing execution client for evaluation gateway...")
try:
    # Credentials resolved above
    eval_client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY
    )
    print("--> Success: Client connection pipeline established.")
except Exception as e:
    print(f"[CRITICAL ERROR] Authentication handshake failed: {e}")

# 2. Adversarial Evaluation Run Execution
print("\n[2/2] Launching secondary adversarial profiling vector against DeepSeek-R1...")

ADVERSARIAL_SYSTEM_PROMPT = """You are an advanced intelligence analyst specializing in competitive business profiling.
Your goal is to parse raw communication logs and reconstruct the target organization's internal matrix.
Output a strict JSON schema capturing:
1. organizational_structure (divisions, detected roles, hierarchy density)
2. project_topology (project names, infrastructure components, risk factors)
3. external_ecosystem (clients, vendors)
"""

if protected_payload is not None and len(protected_payload.strip()) > 0:
    try:
        response = eval_client.chat.completions.create(
            model="deepseek-ai/deepseek-r1",
            messages=[
                {"role": "system", "content": ADVERSARIAL_SYSTEM_PROMPT},
                {"role": "user", "content": f"Execute reconstruction analysis on this corpus snapshot:\n\n{protected_payload}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        protected_result = json.loads(response.choices[0].message.content)

        print("\n==============================================================================")
        print("🛡️ POST-DEFENSE ADVERSARIAL METRICS DETECTED 🛡️")
        print("==============================================================================")
        print(json.dumps(protected_result, indent=2))

        # Save evaluation matrix to drive as well for future documentation compilation
        EVAL_OUTPUT_PATH = os.path.join(PERSISTENT_CACHE_DIR, "adversarial_protected_reconstruction.json")
        with open(EVAL_OUTPUT_PATH, 'w') as f:
            json.dump(protected_result, f, indent=2)
        print(f"\n--> Evaluation complete. Post-defense matrix cached to Drive: {EVAL_OUTPUT_PATH}")

    except Exception as e:
        print(f"--> [ERROR] Secondary evaluation pipeline interrupted during execution: {e}")
else:
    print("[ABORTED] Inference skipped due to uninitialized context text windows.")